<a href="https://colab.research.google.com/github/amitparikh-108/agentic-ai-ik/blob/main/Hybrid_Search_%26_Retrieval_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hybrid Search & Retrieval — Assignment**

## **Problem Statement**

Build a **hybrid search & retrieval system** over biomedical research abstracts using the [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) dataset. Your system must combine **two complementary retrieval strategies** — dense (semantic) and sparse (lexical) — and fuse their results using **Reciprocal Rank Fusion (RRF)** so that the final ranking outperforms either method alone.

You will:

1. Load & clean a subset of PubMedQA  
2. Construct rich, searchable document text from structured fields  
3. Generate **dense** embeddings and **sparse** embeddings
4. Index documents in **Qdrant** with both vector types  
5. Implement a **hybrid search** function that queries both indexes in parallel  
6. Implement **RRF** to merge and re-rank the two result lists  
7. Test and compare all three retrieval modes (dense-only, sparse-only, hybrid)

### Architecture

```
                        ┌─── Dense (BGE-Large) ──────┐
Query ──► Embed Query ──┤                            ├──► RRF Fusion ──► Final Ranked Results         
                        └─── Sparse (SPLADE) ────────┘
```

### Key Concepts

- **Dense retrieval** excels at *semantic* / conceptual matching (e.g., "physical activity" ↔ "exercise")  
- **Sparse retrieval** excels at *exact* / lexical matching (e.g., drug names like "metformin")  
- **Hybrid search** combines both to cover each method's blind spots  
- **RRF** merges ranked lists without needing score normalization — a document found by *both* methods is promoted



## **💡 Tips**




**Document construction matters.** Think carefully about *which* fields to include in the document text and *which* to exclude. The `question` field is what users will search with — including it in the document would allow trivial self-matching and inflate your results.



## **Steps**

In [1]:
!pip install -q qdrant-client fastembed datasets transformers

### Step 1: Setup & Installation
Install the required packages: `qdrant-client`, `fastembed`, `datasets`, `transformers`. Import all necessary modules.

You will need:
- `QdrantClient` and models from `qdrant_client.models` (`Distance`, `SparseVector`, `PointStruct`, `QueryRequest`, `SparseIndexParams`, `SparseVectorParams`, `VectorParams`, `ScoredPoint`)
- `SparseTextEmbedding` and `TextEmbedding` from `fastembed`
- `load_dataset` from `datasets`
- `pandas`, `numpy`, `json`


In [2]:
# Write your code here
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint
)

import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding

print(f"FastEmbed version: {fastembed.__version__}")

FastEmbed version: 0.9.0


### Step 2: Load & Explore the PubMedQA Dataset
Load the `pqa_artificial` split of `qiaojin/PubMedQA`. Print total instances and feature names. Then select a **subset of 100** instances to work with and convert to a pandas DataFrame.


In [3]:
# Load the PubMedQA artificial split
dataset = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

In [4]:
print(f"Total instances: {len(dataset)}")
print(f"Features: {dataset.column_names}")

Total instances: 21127
Features: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


In [6]:
# Work with a subset of 100 for this assignment
dataset = dataset.select(range(100))
print(f"Working with {len(dataset)} instances")

Working with 100 instances


In [7]:
df = dataset.to_pandas()
df.head()

,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) in...,{'contexts': ['Chronic rhinosinusitis (CRS) is...,"As ILC2s are elevated in patients with CRSwNP,...",yes
1,25433161,Does vagus nerve contribute to the development...,{'contexts': ['Phosphatidylethanolamine N-meth...,Neuronal signals via the hepatic vagus nerve c...,yes
2,25445714,Does psammaplin A induce Sirtuin 1-dependent a...,{'contexts': ['Psammaplin A (PsA) is a natural...,PsA significantly inhibited MCF-7/adr cells pr...,yes
3,25431941,Is methylation of the FGFR2 gene associated wi...,{'contexts': ['This study examined links betwe...,We identified a novel biologically plausible c...,yes
4,25432519,Do tumor-infiltrating immune cell profiles and...,{'contexts': ['Tumor microenvironment immunity...,Breast cancer immune cell subpopulation profil...,yes


### Step 3: Data Cleaning & Exploration
Before building documents, inspect the data:
- Check for duplicate `pubid` values  
- Check for missing values and empty strings in `question` and `long_answer`  
- Print the distribution of `final_decision` (yes / no / maybe)

This step ensures you understand what you're working with.


In [9]:
# Check for duplicate pubids
print(f"Total rows:           {len(df)}")
print(f"Unique pubids:        {df['pubid'].nunique()}")
print(f"Duplicate pubids:     {df.duplicated(subset=['pubid']).sum()}")



Total rows:           100
Unique pubids:        100
Duplicate pubids:     0


In [10]:
# Check for duplicate pubids
print(f"Total rows:           {len(df)}")
print(f"Unique pubids:        {df['pubid'].nunique()}")
print(f"Duplicate pubids:     {df.duplicated(subset=['pubid']).sum()}")

print(f"\n--- Missing Values ---")
print(df.isnull().sum())

print(f"\n--- Empty Strings ---")
print(f"Empty questions:      {(df['question'] == '').sum()}")
print(f"Empty long_answers:   {(df['long_answer'] == '').sum()}")

print(f"\n--- Final Decision Distribution ---")
print(df['final_decision'].value_counts())

Total rows:           100
Unique pubids:        100
Duplicate pubids:     0

--- Missing Values ---
pubid             0
question          0
context           0
long_answer       0
final_decision    0
dtype: int64

--- Empty Strings ---
Empty questions:      0
Empty long_answers:   0

--- Final Decision Distribution ---
final_decision
yes    89
no     11
Name: count, dtype: int64


### Step 4: Construct the Document Text
Write a function that combines:
1. **Context paragraphs** — prefixed with their section label, e.g. `[BACKGROUND] ...`  
2. **Long answer** (conclusion) — prefixed with `[CONCLUSION]`  

**Important:** Do **NOT** include the `question` field in the document text — that's what users will query with.

Apply this function to create a new `combined_text` column in your DataFrame.


In [ ]:
# Write your code here

### Step 5: Load Embedding Models
Initialize two models from FastEmbed:
- **Sparse model:** `prithvida/Splade_PP_en_v1` — using `SparseTextEmbedding`
- **Dense model:** `BAAI/bge-large-en-v1.5` — using `TextEmbedding`

Write helper functions `make_sparse_embedding(texts)` and `make_dense_embedding(texts)` that wrap each model's `.embed()` method.


In [ ]:
# Write your code here

### Step 6: Generate Embeddings
Embed all  documents using both models. Store results as new columns `sparse_embedding` and `dense_embedding` in the DataFrame.



In [ ]:
# Write your code here

### Step 7: Create the Qdrant Collection
Initialize an in-memory `QdrantClient`. Create a collection named `"pubmedqa"` configured with:
- A **dense vector** config named `"text-dense"` — size 1024, cosine distance  
- A **sparse vector** config named `"text-sparse"`


In [ ]:
# Write your code here

### Step 8: Prepare & Upload Points
Write a function `make_points(df)` that converts each DataFrame row into a `PointStruct` containing:
- **id:** row index  
- **payload:** `text`, `pubid`, `question`, `final_decision`  
- **vectors:** both `text-dense` (list) and `text-sparse` (`SparseVector` with indices and values)

Then upsert all points into the collection.


In [ ]:
# Write your code here

### Step 9: Implement Hybrid Search
Write a `search(query_text, top_k=10)` function that:
1. Embeds the query with **both** models  
2. Runs dense and sparse queries in **parallel** using `client.query_batch_points()`  
3. Returns both result lists: `[dense_results, sparse_results]`


In [ ]:
# Write your code here

### Step 10: Test Search Queries
Test your search with **different query types** and compare dense vs. sparse results:

For each query, print the top 5 results from both dense and sparse, showing scores and text previews.


In [ ]:
# Write your code here

### Step 11: Implement Reciprocal Rank Fusion (RRF)
Implement two functions:

1. `rank_list(search_result)` — converts Qdrant `ScoredPoint` results into `(id, rank)` pairs (1-indexed)  
2. `rrf(rank_lists, k=60, default_rank=1000)` — takes multiple rank lists and returns a single fused list sorted by RRF score

**RRF formula:** For each item, sum `1 / (k + rank)` across all lists. Items not present in a list get `default_rank`.


In [ ]:
# Write your code here

### Step 12: Apply RRF & Analyze Results
Apply RRF to fuse dense and sparse results from one of your test queries. Print the final ranked list, showing for each result:
- Its rank  
- Whether it came from **BOTH**, **Dense only**, or **Sparse only**  
- A text preview

**Analyze:** Do you see documents promoted that appeared in both lists? Are there any results that only one method found?


In [ ]:
# Write your code here